# Phase 1: Machine Learning — Employee Churn Prediction
**Goal:** Problem Framing, Data Exploration, and Model Selection Planning

---

## Business Use Case
Predict whether an employee is likely to leave the organization voluntarily within the next 6–12 months.

**ML Problem Type:** Binary Classification  
**Target Variable:** `Attrition` — Yes (1) / No (0)  
**Dataset:** IBM HR Analytics Employee Attrition Dataset (Kaggle / UCI)

---

## 0. Install & Import Dependencies

In [ ]:
# Install required packages if not already installed
import subprocess, sys

packages = ["pandas", "numpy", "matplotlib", "seaborn", "scikit-learn", "xgboost", "shap", "imbalanced-learn"]
for pkg in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

print("All packages ready.")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    RocCurveDisplay, ConfusionMatrixDisplay, f1_score
)
from xgboost import XGBClassifier
import shap

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120

print("Libraries imported successfully.")
print(f"Pandas  : {pd.__version__}")
print(f"NumPy   : {np.__version__}")
print(f"Seaborn : {sns.__version__}")

---
## 1. Dataset Acquisition & Loading

**Source:** IBM HR Analytics Employee Attrition dataset — `WA_Fn-UseC_-HR-Employee-Attrition.csv`  
**Origin:** IBM / Watson Analytics sample dataset (publicly available via Kaggle)  
**File loaded locally** — no internet connection required.


In [ ]:
import os

DATA_PATH = "WA_Fn-UseC_-HR-Employee-Attrition.csv"

df = pd.read_csv(DATA_PATH, encoding="utf-8-sig")   # utf-8-sig strips the BOM character
print(f"Dataset loaded: {DATA_PATH}")
print(f"Shape : {df.shape[0]} rows × {df.shape[1]} columns")
df.head(3)


---
## 2. Basic Data Assessment (Exploratory Only)

> Note: Detailed preprocessing (encoding, scaling, imputation) is covered in a separate course.  
> This section performs **exploratory assessment only**.

In [ ]:
# ── Shape and column overview ──────────────────────────────────────────────
print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")
print("\nColumn names:")
print(df.columns.tolist())

In [ ]:
# ── Data types and non-null counts ────────────────────────────────────────
df.info()

In [ ]:
# ── Missing value check ───────────────────────────────────────────────────
missing = df.isnull().sum()
print("Missing values per column:")
print(missing[missing > 0] if missing.any() else "No missing values found.")

In [ ]:
# ── Constant-value columns (zero variance — safe to drop later) ───────────
const_cols = [c for c in df.columns if df[c].nunique() == 1]
print(f"Constant-value columns (to be dropped in preprocessing): {const_cols}")

In [ ]:
# ── Duplicate rows ────────────────────────────────────────────────────────
print(f"Duplicate rows: {df.duplicated().sum()}")

In [ ]:
# ── Numerical summary statistics ──────────────────────────────────────────
df.describe().T.style.background_gradient(cmap="Blues", subset=["mean", "std"])

In [ ]:
# ── Categorical columns and their unique value counts ─────────────────────
cat_cols = df.select_dtypes(include="object").columns.tolist()
print("Categorical columns:")
for c in cat_cols:
    print(f"  {c:30s} → {df[c].unique().tolist()}")

---
## 3. Target Variable Analysis — Class Distribution

In [ ]:
attrition_counts = df["Attrition"].value_counts()
attrition_pct    = df["Attrition"].value_counts(normalize=True) * 100

print("Attrition Distribution:")
summary = pd.DataFrame({"Count": attrition_counts, "Percentage": attrition_pct.round(2)})
print(summary.to_string())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Bar chart
colors = ["#4C72B0", "#DD8452"]
axes[0].bar(attrition_counts.index, attrition_counts.values, color=colors, edgecolor="white", width=0.5)
for i, (val, pct) in enumerate(zip(attrition_counts.values, attrition_pct.values)):
    axes[0].text(i, val + 15, f"{val}\n({pct:.1f}%)", ha="center", fontsize=11, fontweight="bold")
axes[0].set_title("Attrition Count", fontweight="bold")
axes[0].set_xlabel("Attrition")
axes[0].set_ylabel("Count")
axes[0].set_ylim(0, attrition_counts.max() * 1.2)

# Pie chart
axes[1].pie(attrition_counts.values, labels=attrition_counts.index, autopct="%1.1f%%",
            colors=colors, startangle=90, wedgeprops=dict(edgecolor="white", linewidth=2))
axes[1].set_title("Attrition Split", fontweight="bold")

plt.suptitle("Target Variable: Employee Attrition", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("fig1_attrition_distribution.png", bbox_inches="tight")
plt.show()
print("Class imbalance detected — will require SMOTE or class_weight='balanced' during modeling.")

---
## 4. Exploratory Data Analysis (EDA)

### 4.1 Attrition by Key Categorical Variables

In [ ]:
cat_features = ["OverTime", "Department", "JobRole", "BusinessTravel", "MaritalStatus"]

fig, axes = plt.subplots(3, 2, figsize=(14, 14))
axes = axes.flatten()

for i, feat in enumerate(cat_features):
    ct = df.groupby([feat, "Attrition"]).size().unstack(fill_value=0)
    ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100
    ct_pct.plot(kind="bar", ax=axes[i], color=["#4C72B0", "#DD8452"], edgecolor="white",
                width=0.65, legend=(i == 0))
    axes[i].set_title(f"Attrition by {feat}", fontweight="bold")
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel("Percentage (%)")
    axes[i].tick_params(axis="x", rotation=30)
    axes[i].yaxis.set_major_formatter(mticker.PercentFormatter())
    if i == 0:
        axes[i].legend(title="Attrition", loc="upper right")

axes[-1].set_visible(False)
plt.suptitle("Attrition Rate by Categorical Features", fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("fig2_attrition_by_category.png", bbox_inches="tight")
plt.show()

### 4.2 Attrition by Key Numerical Variables

In [ ]:
num_features = ["Age", "MonthlyIncome", "YearsAtCompany", "DistanceFromHome",
                "TotalWorkingYears", "YearsInCurrentRole"]

fig, axes = plt.subplots(3, 2, figsize=(14, 12))
axes = axes.flatten()

for i, feat in enumerate(num_features):
    for label, color in zip(["No", "Yes"], ["#4C72B0", "#DD8452"]):
        data = df[df["Attrition"] == label][feat]
        axes[i].hist(data, bins=20, alpha=0.6, color=color, label=f"Attrition={label}",
                     edgecolor="white")
    axes[i].set_title(f"Distribution of {feat}", fontweight="bold")
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel("Frequency")
    axes[i].legend()

plt.suptitle("Numerical Feature Distributions by Attrition", fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("fig3_numerical_distributions.png", bbox_inches="tight")
plt.show()

### 4.3 Satisfaction Scores vs Attrition (Box Plots)

In [ ]:
satisfaction_cols = [
    "JobSatisfaction", "EnvironmentSatisfaction",
    "WorkLifeBalance", "RelationshipSatisfaction"
]

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
axes = axes.flatten()

for i, col in enumerate(satisfaction_cols):
    sns.boxplot(data=df, x="Attrition", y=col, ax=axes[i],
                palette={"No": "#4C72B0", "Yes": "#DD8452"},
                width=0.5, linewidth=1.5)
    axes[i].set_title(f"{col} vs Attrition", fontweight="bold")
    axes[i].set_xlabel("Attrition")
    axes[i].set_ylabel(col)

plt.suptitle("Satisfaction Scores by Attrition Status", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.savefig("fig4_satisfaction_boxplots.png", bbox_inches="tight")
plt.show()

### 4.4 Correlation Heatmap (Numerical Features)

In [ ]:
# Encode target for correlation purposes only
df_corr = df.copy()
df_corr["Attrition_enc"] = (df_corr["Attrition"] == "Yes").astype(int)

num_cols = df_corr.select_dtypes(include=["int64", "float64"]).columns.tolist()
corr_matrix = df_corr[num_cols].corr()

# Focus on correlation with target
target_corr = (
    corr_matrix["Attrition_enc"]
    .drop("Attrition_enc")
    .sort_values(key=abs, ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Full heatmap
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, ax=axes[0], cmap="coolwarm", center=0,
            linewidths=0.4, annot=False, fmt=".1f", square=True,
            cbar_kws={"shrink": 0.7})
axes[0].set_title("Feature Correlation Heatmap", fontweight="bold")
axes[0].tick_params(axis="x", rotation=90, labelsize=7)
axes[0].tick_params(axis="y", labelsize=7)

# Correlation with target bar chart
colors_corr = ["#DD8452" if v > 0 else "#4C72B0" for v in target_corr.values]
axes[1].barh(target_corr.index, target_corr.values, color=colors_corr, edgecolor="white")
axes[1].axvline(0, color="black", linewidth=0.8, linestyle="--")
axes[1].set_title("Feature Correlation with Attrition", fontweight="bold")
axes[1].set_xlabel("Pearson Correlation Coefficient")
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig("fig5_correlation_heatmap.png", bbox_inches="tight")
plt.show()

print("Top 5 features most correlated with Attrition:")
print(target_corr.head(5).to_string())

### 4.5 Attrition Rate by Job Role (Ranked)

In [ ]:
role_attrition = (
    df.groupby("JobRole")["Attrition"]
    .apply(lambda x: (x == "Yes").mean() * 100)
    .sort_values(ascending=False)
    .reset_index()
)
role_attrition.columns = ["JobRole", "AttritionRate"]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.barh(role_attrition["JobRole"], role_attrition["AttritionRate"],
               color=sns.color_palette("Reds_r", len(role_attrition)),
               edgecolor="white")
for bar, val in zip(bars, role_attrition["AttritionRate"]):
    ax.text(val + 0.3, bar.get_y() + bar.get_height() / 2,
            f"{val:.1f}%", va="center", fontsize=10)

ax.set_xlabel("Attrition Rate (%)")
ax.set_title("Attrition Rate by Job Role", fontweight="bold")
ax.set_xlim(0, role_attrition["AttritionRate"].max() + 8)
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("fig6_attrition_by_jobrole.png", bbox_inches="tight")
plt.show()

---
## 5. Minimal Feature Encoding for Model Prototyping

> Only Label Encoding is applied here for prototyping.  
> Full preprocessing (One-Hot Encoding, scaling, SMOTE) belongs to the preprocessing course.

In [ ]:
df_model = df.drop(columns=const_cols, errors="ignore").copy()

le = LabelEncoder()
for col in df_model.select_dtypes(include="object").columns:
    df_model[col] = le.fit_transform(df_model[col])

X = df_model.drop(columns=["Attrition"])
y = df_model["Attrition"]

print(f"Features : {X.shape[1]}")
print(f"Samples  : {X.shape[0]}")
print(f"Target distribution:\n{y.value_counts().to_string()}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train size : {X_train.shape[0]}  |  Test size : {X_test.shape[0]}")
print(f"Train class distribution:\n{y_train.value_counts().to_string()}")

---
## 6. Algorithm 1 — Random Forest Classifier

**Justification:**
- Handles mixed numerical and categorical (encoded) features
- Provides feature importance — interpretable for HR stakeholders
- Robust to outliers and class imbalance via `class_weight='balanced'`
- No distributional assumptions

In [ ]:
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)

rf_preds  = rf_model.predict(X_test)
rf_probas = rf_model.predict_proba(X_test)[:, 1]

rf_f1  = f1_score(y_test, rf_preds, pos_label=1)
rf_auc = roc_auc_score(y_test, rf_probas)

print("=" * 45)
print("  Random Forest — Evaluation Results")
print("=" * 45)
print(f"  F1-Score (Attrition=Yes) : {rf_f1:.4f}")
print(f"  AUC-ROC                  : {rf_auc:.4f}")
print()
print(classification_report(y_test, rf_preds, target_names=["No Attrition", "Attrition"]))

In [ ]:
# Stratified Cross-Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(rf_model, X, y, cv=cv, scoring="roc_auc", n_jobs=-1)
print(f"Random Forest — 5-Fold CV AUC-ROC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"Fold-wise scores: {[round(s, 4) for s in cv_scores]}")

In [ ]:
# Random Forest — Confusion Matrix & ROC Curve
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test, rf_preds, display_labels=["No Attrition", "Attrition"],
    colorbar=False, cmap="Blues", ax=axes[0]
)
axes[0].set_title("Random Forest — Confusion Matrix", fontweight="bold")

RocCurveDisplay.from_predictions(
    y_test, rf_probas, name=f"Random Forest (AUC={rf_auc:.3f})",
    color="#4C72B0", ax=axes[1]
)
axes[1].plot([0, 1], [0, 1], "k--", linewidth=1)
axes[1].set_title("Random Forest — ROC Curve", fontweight="bold")

plt.tight_layout()
plt.savefig("fig7_rf_evaluation.png", bbox_inches="tight")
plt.show()

In [ ]:
# Random Forest — Feature Importance (Top 15)
fi = pd.Series(rf_model.feature_importances_, index=X.columns)
fi_top = fi.sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(fi_top.index[::-1], fi_top.values[::-1],
               color=sns.color_palette("Blues_r", 15), edgecolor="white")
for bar, val in zip(bars, fi_top.values[::-1]):
    ax.text(val + 0.001, bar.get_y() + bar.get_height() / 2,
            f"{val:.3f}", va="center", fontsize=9)

ax.set_xlabel("Feature Importance (Gini)")
ax.set_title("Random Forest — Top 15 Feature Importances", fontweight="bold")
plt.tight_layout()
plt.savefig("fig8_rf_feature_importance.png", bbox_inches="tight")
plt.show()

---
## 7. Algorithm 2 — XGBoost Classifier

**Justification:**
- State-of-the-art performance on structured/tabular data
- Handles class imbalance via `scale_pos_weight`
- Built-in L1/L2 regularization — prevents overfitting on small datasets
- Compatible with SHAP for explainability

In [ ]:
scale_pos = int((y_train == 0).sum() / (y_train == 1).sum())

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    scale_pos_weight=scale_pos,
    use_label_encoder=False,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)
xgb_model.fit(X_train, y_train)

xgb_preds  = xgb_model.predict(X_test)
xgb_probas = xgb_model.predict_proba(X_test)[:, 1]

xgb_f1  = f1_score(y_test, xgb_preds, pos_label=1)
xgb_auc = roc_auc_score(y_test, xgb_probas)

print("=" * 45)
print("  XGBoost — Evaluation Results")
print("=" * 45)
print(f"  F1-Score (Attrition=Yes) : {xgb_f1:.4f}")
print(f"  AUC-ROC                  : {xgb_auc:.4f}")
print()
print(classification_report(y_test, xgb_preds, target_names=["No Attrition", "Attrition"]))

In [ ]:
cv_scores_xgb = cross_val_score(xgb_model, X, y, cv=cv, scoring="roc_auc", n_jobs=-1)
print(f"XGBoost — 5-Fold CV AUC-ROC: {cv_scores_xgb.mean():.4f} ± {cv_scores_xgb.std():.4f}")
print(f"Fold-wise scores: {[round(s, 4) for s in cv_scores_xgb]}")

In [ ]:
# XGBoost — Confusion Matrix & ROC Curve
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ConfusionMatrixDisplay.from_predictions(
    y_test, xgb_preds, display_labels=["No Attrition", "Attrition"],
    colorbar=False, cmap="Oranges", ax=axes[0]
)
axes[0].set_title("XGBoost — Confusion Matrix", fontweight="bold")

RocCurveDisplay.from_predictions(
    y_test, xgb_probas, name=f"XGBoost (AUC={xgb_auc:.3f})",
    color="#DD8452", ax=axes[1]
)
axes[1].plot([0, 1], [0, 1], "k--", linewidth=1)
axes[1].set_title("XGBoost — ROC Curve", fontweight="bold")

plt.tight_layout()
plt.savefig("fig9_xgb_evaluation.png", bbox_inches="tight")
plt.show()

---
## 8. SHAP Explainability — XGBoost

In [ ]:
explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_test, plot_type="bar", max_display=15,
                  show=False, color="#DD8452")
plt.title("SHAP Feature Importance (XGBoost)", fontweight="bold", fontsize=13)
plt.tight_layout()
plt.savefig("fig10_shap_bar.png", bbox_inches="tight")
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_test, max_display=15, show=False)
plt.title("SHAP Summary Plot — Impact on Attrition Prediction", fontweight="bold", fontsize=13)
plt.tight_layout()
plt.savefig("fig11_shap_summary.png", bbox_inches="tight")
plt.show()

---
## 9. Model Comparison Summary

In [ ]:
comparison = pd.DataFrame({
    "Model": ["Random Forest", "XGBoost"],
    "F1-Score (Attrition)": [round(rf_f1, 4), round(xgb_f1, 4)],
    "AUC-ROC": [round(rf_auc, 4), round(xgb_auc, 4)],
    "CV AUC Mean": [round(cv_scores.mean(), 4), round(cv_scores_xgb.mean(), 4)],
    "CV AUC Std": [round(cv_scores.std(), 4), round(cv_scores_xgb.std(), 4)],
})

print("Model Comparison:")
print(comparison.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
metrics = ["F1-Score (Attrition)", "AUC-ROC"]
colors  = ["#4C72B0", "#DD8452"]

for i, metric in enumerate(metrics):
    bars = axes[i].bar(comparison["Model"], comparison[metric],
                       color=colors, edgecolor="white", width=0.45)
    for bar, val in zip(bars, comparison[metric]):
        axes[i].text(bar.get_x() + bar.get_width() / 2, val + 0.005,
                     f"{val:.4f}", ha="center", fontweight="bold", fontsize=11)
    axes[i].set_title(metric, fontweight="bold")
    axes[i].set_ylim(0, 1.0)
    axes[i].set_ylabel("Score")
    axes[i].axhline(0.75, color="red", linestyle="--", linewidth=1, label="Business threshold")
    axes[i].legend(fontsize=8)

plt.suptitle("Model Comparison — Random Forest vs XGBoost", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("fig12_model_comparison.png", bbox_inches="tight")
plt.show()

---
## 10. Business KPIs & Key Insights

In [ ]:
total        = len(df)
attrition_n  = (df["Attrition"] == "Yes").sum()
attrition_rt = attrition_n / total * 100

avg_salary   = df["MonthlyIncome"].mean() * 12
replace_cost = avg_salary * 1.5
total_cost   = attrition_n * replace_cost

# Recall for attrition class
from sklearn.metrics import recall_score
rf_recall  = recall_score(y_test, rf_preds,  pos_label=1)
xgb_recall = recall_score(y_test, xgb_preds, pos_label=1)

print("=" * 55)
print("  Business KPI Dashboard")
print("=" * 55)
print(f"  Total Employees          : {total}")
print(f"  Attrition Count          : {attrition_n}")
print(f"  Attrition Rate           : {attrition_rt:.1f}%")
print(f"  Avg Annual Salary        : ${avg_salary:,.0f}")
print(f"  Est. Replacement Cost    : ${replace_cost:,.0f} per employee")
print(f"  Total Attrition Cost     : ${total_cost:,.0f}")
print()
print(f"  Business Recall Threshold: ≥ 0.75 (catch 75%+ of churners)")
print(f"  RF  Recall (Attrition)   : {rf_recall:.4f}  {'PASS' if rf_recall >= 0.75 else 'FAIL'}")
print(f"  XGB Recall (Attrition)   : {xgb_recall:.4f}  {'PASS' if xgb_recall >= 0.75 else 'FAIL'}")
print("=" * 55)

---
## 11. Concept Note Summary

| Item | Detail |
|---|---|
| **Business Use Case** | Employee Churn Prediction |
| **ML Problem Type** | Binary Classification |
| **Dataset** | IBM HR Analytics — 1,470 rows × 35 features |
| **Class Imbalance** | 83.9% No / 16.1% Yes — handled via `class_weight='balanced'` / `scale_pos_weight` |
| **Algorithm 1** | Random Forest — interpretable, robust, feature importance |
| **Algorithm 2** | XGBoost — high accuracy, regularisation, SHAP-compatible |
| **Why not DL** | Dataset too small; no unstructured data |
| **Primary KPI** | Reduce attrition rate from ~16% → ~10–12% |
| **Eval Metrics** | F1-Score + AUC-ROC; Recall ≥ 0.75 for attrition class |
| **Tools** | Scikit-learn, XGBoost, SHAP, Pandas, Seaborn, imbalanced-learn |

---
*Phase 1 complete. Detailed preprocessing, feature engineering, hyperparameter tuning, and final model evaluation are covered in subsequent phases.*